# Model MDS geometry, grouped

The per-model MDS plates answer "what did this base model's finetuning do".
These answer the questions that need more than one model in the same picture:
whether size changes the answer, whether an open base sits where a proprietary
one sits, and whether the culture-MLLM-finetuned variants agree with each other.

Every re-run first rebuilds **every** per-model culture MDS plate — matched,
comparison, all-countries and the outcomes row — over every model, variant and
question, then draws three cross-model families. That refresh is why the
notebook exists: `culture-mds` is its own command, so a model swept after the
last invocation of it simply had no MDS figures. The sweep calls it now too;
this is the by-hand path, and the one that also writes the grouped plates.

Three families, each one shared embedding on one shared scale, so every panel of
a plate is directly comparable:

- `fig_culture_mds_tier_{small,mid,big}` — one column per model in the size
  tier, rows NTP and FA, both variants in every panel, over the German cells.
- `fig_culture_mds_base_families` — the as-released variant of every model, open
  first then served, over the cells all of them share in all five countries.
  Full answers only: the served endpoints return no next-token probabilities, so
  a plate that includes them has no NTP half to draw.
- `fig_culture_mds_finetuned` — the six `german` variants side by side over the
  German cells.

| encoding | meaning |
|---|---|
| yellow triangles | WVS respondents over the plate's shared sample |
| dark circle, pale square | Mixtral archived and fresh — the run-to-run noise floor |
| hollow diamond | the as-released variant |
| solid marker | the `german` culture-MLLM-finetuned variant |
| panel title | the model |
| strap line | mode · cells · every series' mean nEMD over that sample |

The strap-line nEMD is over the shared sample only, never the run's overall
error, and it is computed from the response distributions rather than from the
coordinates — so it does not move when the embedding does.

A run that never happened contributes no panel and no row. `luna` and `sol` have
not run at all and `terra` carries happiness and politics only, so those columns
appear as the runs land; a model missing either variant sits its tier plate out.

These embeddings hold every drawn series at once, which is why the survey block
enters them once rather than once per series the way the per-model plates stack
it. Repeating it would triple the largest embedding here to re-centre a picture
whose reported numbers never see it.

Writes `figures/culture/<question>/fig_culture_mds_tier_<tier>`,
`fig_culture_mds_base_families` and `fig_culture_mds_finetuned`, `.png` and
`.pdf`, plus `outputs/culture/culture_mds_groups.csv`. The refresh also rewrites
`outputs/culture/<question>/culture_mds_matched.csv`, which stays complete only
because it runs over every model — a hand-run `culture-mds --models X` replaces
that table with X alone.

A full four-question pass is tens of minutes. Every driver takes a question
list, so `tier_plates(["d_happy"])` is the quick look.

`.venv` has no jupyter and a bare `uv run` must not be used here — it destroys
the hand-built sm_120 `llama-cpp-python` wheel. Committed without cell outputs.

In [1]:
import pandas as pd

from culture.mds_groups import (
    GROUPS_TABLE,
    base_families_plates,
    finetuned_plates,
    refresh_model_plates,
    tier_plates,
)
from machine_bias_reproduction.io_utils import write_csv

## Refreshing the per-model plates

Every model, every variant, every question, with the all-countries plate and the
outcomes row included. The default model set is not a convenience here: it is
what keeps `culture_mds_matched.csv` from being replaced by whichever subset was
asked for.

In [2]:
files = refresh_model_plates()
print(len(files), "files")

312 files


## The tier plates

Three buckets declared in `culture.registry.SIZE_TIERS`, so a new base model
picks up a column by being added there and nowhere else. The served models have
no published size and sit in none of them.

Each panel holds one model's two variants over the same German cells, hollow
against solid, on the plate's one scale.

In [3]:
tier_files, tier_table = tier_plates()
print(len(tier_files), "files")

24 files


## Open against proprietary — as-released variants, full answers only

The served models are never culture-finetuned, so the as-released variant is the
only one they have and this is the only plate they can appear in. It is full
answers throughout, including for the open models that do have an NTP half, so
that every panel is the same measurement.

The sample is the intersection of the cells every drawn run retained, over all
five countries — an as-released model has no home culture, so restricting to Germany
would answer a question nobody asked. One shrunken run shrinks the sample for
every panel; that is the price of one comparable measurement.

In [4]:
base_files, base_table = base_families_plates()
print(len(base_files), "files")

8 files


## The german variants side by side

The same German cells, the finetuned variant of every open base at once. The
per-model comparison plate says whether one family's finetuning moved toward its
own respondents; this says whether the families agree.

In [5]:
tuned_files, tuned_table = finetuned_plates()
print(len(tuned_files), "files")

8 files


## The ledger

One repo-wide CSV rather than one per question, rebuilt whole on every run, the
way `population_adaptability.csv` is — `plate` and `question` slice it. The two
Mixtral rows carry no model key: their value is identical in every panel of a
shared embedding, so they are kept once per plate and mode.

In [6]:
table = pd.concat([tier_table, base_table, tuned_table], ignore_index=True)
write_csv(table, GROUPS_TABLE)
print(len(table), "rows")

244 rows


## Reading the plates

Inside a panel, hollow against solid is what the culture-MLLM finetuning moved,
and both sit on the plate's one scale, so the size of that move is comparable
with every other panel. Across a row the columns differ only in the model.

Any gap has to clear the gap between the two Mixtral clouds. They are one model
scored twice, so whatever separates them is run-to-run noise, and a difference
smaller than that is not a finding.

On the base-families plate, a served column further from the yellow cloud than
every open column is a result about that family, not about capacity — the served
sizes are unpublished, which is exactly why they are in no tier. The tier plates
are where a size claim can be made.

A cloud drawn as a tight arc rather than a spread is a variant answering nearly
the same way for every subpopulation it was asked about.
`population_adaptability.ipynb` measures that directly; here it is a shape, not
a number.

Nothing here is a significance claim.